# Nghe — build the audio

Run the cells top to bottom. Everything happens on Kaggle's machines, not yours.

This builds **two accents** — Southern and Northern — as separate folders of clips, `audio/south/` and `audio/north/`. The app lets you choose which accent to practise with.

> These cells also run on Google Colab unchanged — the token cell detects either platform — but the setup instructions below are for Kaggle.

## 1. Turn on GPU, internet, and your token

All three are in the **right-hand sidebar** (open it with the **`<`** arrow or **Settings**):

- **Accelerator → GPU T4 x2** (or **GPU P100**). We only use one GPU.
- **Internet → On.** Needed to install VieNeu, clone your repo and push back. Kaggle requires a phone-verified account to enable it.
- **Add-ons → Secrets →** add a secret named `GITHUB_TOKEN`, paste your token, and tick it to attach it to this notebook.

The token is a GitHub fine-grained personal access token with **Contents: Read and write** on `councilgritter/nghe` (GitHub → Settings → Developer settings → Fine-grained tokens).

## 2. Get the repo

This reads your token from Secrets, clones the repo into the working folder, and moves into it. The token is never printed. Re-running just pulls the latest.

In [ ]:
import os, subprocess

REPO  = 'councilgritter/nghe'
EMAIL = 'sam.saunders96@gmail.com'
NAME  = 'councilgritter'

def get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        return os.environ.get('GITHUB_TOKEN')

TOKEN = get_token()
assert TOKEN, 'No GITHUB_TOKEN found — add it under Add-ons -> Secrets and attach it'

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
REPO_DIR = os.path.join(WORK, 'nghe')
url = f'https://{TOKEN}@github.com/{REPO}.git'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=False)
else:
    subprocess.run(['git', 'clone', url, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'config', 'user.email', EMAIL], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'config', 'user.name', NAME], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url], check=True)
os.chdir(REPO_DIR)
print('working in', REPO_DIR)
print(os.listdir('.'))

## 3. Install VieNeu

VieNeu is the Vietnamese text-to-speech model; librosa and soundfile trim and normalise the clips. This takes a couple of minutes the first time. ffmpeg is already on Kaggle's image; the check below just confirms it.

In [ ]:
!pip install -q vieneu librosa soundfile
!ffmpeg -version | head -1

## 4. See the voices

This lists the built-in voices. The description tells you the accent. You will pick **one voice per accent** in the next step — Northern and Southern both have plenty to choose from.

In [ ]:
from vieneu import Vieneu
tts = Vieneu()
for desc, name in tts.list_preset_voices():
    print(f'{name:24s} {desc}')

## 5. Audition the voices

Set a voice and the two speeds for each accent below, then run the cell that follows to hear 40 of the commonest syllables in each — made by the real pipeline, so what you hear is what the full run produces. Iterate: change a voice or a speed, list the region in `RESET` so its samples are remade, and re-run. Nothing here is committed.

- **VOICES** — a voice name from the list above, per accent.
- **SPEED** — tempo of the level tones (ngang, huyền, sắc, nặng). `1.0` is natural; `0.9` is a touch slower for learners.
- **DIP** — tempo of the two contour tones, hỏi and ngã; slightly slower helps the ear track their shape.
- **RESET** — regions to wipe and remake, e.g. `['north']` after you change the Northern voice or speed.

Each clip is one syllable said on its own, trimmed, loudness-matched, padded with half a second of silence, and slowed. The app's *Slower* button and replay sit on top of this.

In [ ]:
VOICES = {
    'south': 'Adam',
    'north': 'Ngọc Huyền',
}
SPEED = {          # level tones
    'south': 0.9,
    'north': 0.9,
}
DIP = {            # hỏi and ngã, a little slower
    'south': 0.8,
    'north': 0.8,
}
RESET = []         # e.g. ['north'] to wipe a region and remake it

In [ ]:
import glob, csv, os, shutil, subprocess, sys
import IPython.display as ipd

names = {r['clip_id']: r['syllable'] for r in
         csv.DictReader(open('vietnamese_clip_manifest.csv', encoding='utf-8-sig'))}

def generate(region, limit=None):
    if region in RESET:
        shutil.rmtree(f'audio/{region}', ignore_errors=True)
        print('wiped audio/' + region)
    cmd = [sys.executable, 'generate_clips.py', '--region', region,
           '--voice', VOICES[region], '--speed', str(SPEED[region]),
           '--speed-dip', str(DIP[region]), '--used-in', 'data.json']
    if limit: cmd += ['--limit', str(limit)]
    subprocess.run(cmd, check=True)

for region in VOICES:
    print('=====', region, '-', VOICES[region], '@', SPEED[region], '/', DIP[region])
    generate(region, limit=40)
    for f in sorted(glob.glob(f'audio/{region}/*.mp3'))[:10]:
        cid = os.path.basename(f)[:-4]
        print(cid, names.get(cid))
        ipd.display(ipd.Audio(f))

RESET = []   # clear so a re-run does not wipe unless you ask again

## 6. The full run

Once you are happy with the voices and speeds, this makes every clip the app plays (about 6,000 per accent), one accent at a time. It is **resumable** — if the session drops, re-run and it skips what is already done.

Each accent is committed and pushed to GitHub as it finishes, so completed accents are safe. On Kaggle a GPU session runs for hours, so both accents usually fit in one sitting; `TODO` lets you do one at a time if you prefer.

**Note:** work in progress lives only in this session until an accent finishes and pushes. If Kaggle disconnects mid-accent, that accent restarts from scratch on the next run (finished accents are safe on GitHub).

In [ ]:
TODO = ['south', 'north']

def git(*args, check=True):
    return subprocess.run(['git', *args], check=check)

def publish(message, *paths):
    git('add', *paths)
    if git('diff', '--cached', '--quiet', check=False).returncode == 0:
        print('nothing new to commit'); return
    git('-c', 'commit.gpgsign=false', 'commit', '-m', message)
    git('pull', '--rebase', check=False)
    git('push', 'origin', 'HEAD:main')

for region in TODO:
    print('=====', region, '-', VOICES[region], '@', SPEED[region], '/', DIP[region])
    generate(region)
    print(len(glob.glob(f'audio/{region}/*.mp3')), 'clips in', region)
    publish(f'Add {region} clips', f'audio/{region}')

## 7. Done

Both accents are on GitHub now, and the live site updates within a minute of each push. Open it on your phone and have a native Vietnamese speaker check the clips — that is the real test.

**Removing a bad clip later.** Delete `audio/<accent>/<clip_id>.mp3` and push (on your computer with GitHub Desktop, or here by re-running this notebook). The app falls back to the browser voice for anything missing, so nothing breaks.

**Re-running to change a voice or speed.** Edit `VOICES`/`SPEED` in step 5, add that region to `RESET`, and re-run steps 5 and 6. Step 6 only makes what is missing, so `RESET` is how you force a full remake.

**Optional automated check.** `qc_clips.py` transcribes clips back with PhoWhisper if you ever want a rough machine pass — `!python qc_clips.py --region south` — but it flags a lot of false positives on single syllables, so trust human ears first.